# 02 — Prompt, few-shot e piloto no dev

Continuação do notebook 01. Aqui montamos o **prompt** que será usado no teste, escolhemos os **exemplos few-shot**, contamos os **tokens totais de entrada** e rodamos um **piloto no dev** para validar o prompt e o parser antes de tocar no teste.

| Seção | O que faz | Precisa de GPU? |
|---|---|---|
| 1 | Dados e verificações V1–V4, V6 (a partir de `src/finrag.py`) | não |
| 2 | Exemplos few-shot do treino e verificação **V5** | não |
| 3 | Prompt completo de uma pergunta | não |
| 4 | Tokens totais de entrada, por modelo × representação | não |
| 5 | **Piloto no dev** com os dois modelos | **só se o piloto ainda não existir** |
| 6 | Relatório do piloto e verificação **V7** (falha de formato < 5% com R1) | não |
| 7 | Tempo estimado para o teste | não |
| 8 | **Congelamento** do prompt e da configuração (`frozen_config.json`) | não |

**Regra:** todo ajuste de prompt é feito aqui, olhando só o dev. O notebook 03 só roda o teste com a configuração congelada na seção 8.

**Antes de rodar no Colab:** o repositório atualizado no GitHub — este notebook clona o repositório para importar `src/finrag.py`. Os dois modelos do estudo (Qwen3-8B e Phi-4-mini) são abertos: não é preciso token do Hugging Face.

**Checkpoints:** tudo o que é gerado fica na pasta `NLP - Checkpoints/results` do Google Drive (o nome da pasta está em `F.DRIVE_FOLDER`; para usar outra, defina a variável de ambiente `FINRAG_DRIVE_FOLDER`). Se a sessão cair, abra o notebook de novo e rode tudo: o que já foi gravado é lido do Drive e pulado.

In [ ]:
# Ambiente: repositório, módulo compartilhado e pasta de resultados
import os, sys, subprocess, pathlib, json, platform, datetime, random, gc

REPO_URL = "https://github.com/Arthur-Marques-IA/NLP_Fin-RAG.git"

def find_repo():
    here = pathlib.Path.cwd()
    for p in (here, here.parent, here / "NLP_Fin-RAG"):
        if (p / "src" / "finrag.py").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "NLP_Fin-RAG"], check=True)
    return here / "NLP_Fin-RAG"

REPO = find_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
import finrag as F
import pandas as pd

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if os.environ.get("FINRAG_RESULTS_DIR"):
    RESULTS_DIR = pathlib.Path(os.environ["FINRAG_RESULTS_DIR"])
elif IN_COLAB:
    drive.mount("/content/drive")
    RESULTS_DIR = F.drive_results_dir(os.environ.get("FINRAG_DRIVE_FOLDER", F.DRIVE_FOLDER))
else:
    RESULTS_DIR = REPO / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / "data"
FROZEN_PATH = RESULTS_DIR / "frozen_config.json"

try:
    HF_TOKEN = userdata.get("HF_TOKEN") if IN_COLAB else os.environ.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Repositório:", REPO, "| commit:", F.git_sha(REPO))
print("Resultados em:", RESULTS_DIR)
print("Modelos do estudo:", list(F.MODELS), "| HF_TOKEN:", "definido" if HF_TOKEN else "não definido (não é necessário)")

In [ ]:
# GPU e bibliotecas
import torch, transformers

HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_properties(0).name if HAS_GPU else None
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_GPU else 0.0
print("Python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", f"{GPU_NAME} ({GPU_GB:.1f} GB)" if HAS_GPU else "nenhuma — as seções 1 a 4 rodam; o piloto (seção 5) precisa de GPU")

## Configuração

- **`PRECISION`** vale para os **dois modelos** (seção 3 da metodologia) e fica **congelada** na seção 8. Ordem de decisão: a do `frozen_config.json`, se existir; senão, a do piloto já gravado em `pilot_dev/`; senão, `"auto"` escolhe `bf16` se a GPU tiver 22 GB ou mais (L4, A100, G4) e `4bit` (NF4) caso contrário (T4).
- Se `pilot_dev/` já tem o piloto completo do prompt atual, **nenhum modelo é carregado**: o notebook só relê os arquivos, confere a V7 e congela. Nesse caso ele roda em CPU.
- Se `pilot_dev/` tem arquivos de **outro prompt ou outra precisão**, o notebook para antes de gerar qualquer coisa.
- **`ALLOW_NEW_PILOT = False`** impede gerar um piloto por engano: se `pilot_dev/` estiver vazia ou incompleta, o notebook para e avisa, em vez de gastar GPU. O motivo mais comum é a pasta errada do Drive, ou a pasta do piloto com outro nome.
- **`BATCH_SIZE`** também é congelado, porque a composição dos lotes faz parte da reprodutibilidade. Se faltar memória, o lote é dividido ao meio automaticamente (e o tamanho usado fica registrado).
- **`SAVE_EVERY`** é o intervalo de gravação, em lotes: `1` grava no Drive a cada lote (4 perguntas), que é o mais seguro. Não muda os resultados; só define quanto se perde se a sessão cair de repente. A barra de progresso de cada condição mostra perguntas feitas, ritmo, tempo restante e quantas já estão gravadas.
- O piloto usa `PILOT_N` perguntas do dev em **R1** (é a base da V7) e as primeiras `PILOT_OTHER_REPS_N` nas outras cinco representações, para ver se alguma delas quebra o formato da resposta.

In [ ]:
PRECISION = "bf16"                 # precisão do estudo. "auto" decide pela GPU; "4bit" só para testes na T4
MODELS_TO_RUN = list(F.MODELS)     # qwen3-8b, phi4-mini
BATCH_SIZE = 4
SAVE_EVERY = 1                     # grava no Drive a cada N lotes (1 = a cada 4 perguntas)
ALLOW_NEW_PILOT = False            # o piloto do estudo já existe; True só para gerar um piloto novo de propósito
PILOT_N = 150
PILOT_SEED = 7
PILOT_OTHER_REPS_N = 30
V7_MAX_NO_ANSWER = 0.05            # V7a: saídas terminadas sem a linha "Answer:" (falha de formato do prompt)
V7_MIN_ANSWERED = 0.90             # V7b: saídas com resposta legível

In [ ]:
# Precisão efetiva: a do congelamento anterior, se houver; senão, decidida pela GPU
if FROZEN_PATH.exists():
    PRECISION = F.load_json(FROZEN_PATH)["precision"]
    print("Precisão mantida do frozen_config.json:", PRECISION)
elif F.pilot_state(RESULTS_DIR):
    found = sorted({p for _, p in F.pilot_state(RESULTS_DIR)})
    assert len(found) == 1, f"Há piloto em mais de uma precisão em {RESULTS_DIR / 'pilot_dev'}: {found}. Deixe só uma."
    PRECISION = found[0]
    print("Precisão mantida do piloto já gravado:", PRECISION)
elif PRECISION == "auto":
    PRECISION = ("bf16" if GPU_GB >= 22 else "4bit") if HAS_GPU else None
    print("Precisão escolhida pela GPU:", PRECISION)
if PRECISION == "4bit":
    try:
        import bitsandbytes
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)

## 1. Dados e verificações

As representações, a seleção de linhas e o parser vêm de `src/finrag.py`, que é uma cópia do notebook 01 (conferida por `tests/test_consistencia.py`). Reexecutamos aqui as verificações V1–V4 e V6 e a escolha de *k*, para garantir que o que vai para o modelo é o que foi verificado.

In [ ]:
train = F.load_finqa("train", DATA_DIR)
dev = F.load_finqa("dev", DATA_DIR)
test = F.load_finqa("test", DATA_DIR)
print(f"treino: {len(train)} | dev: {len(dev)} | teste: {len(test)}")

k_dev, rates = F.choose_k(dev)
print("Preservação do gabarito no dev por k:", {k: f"{r:.1%}" for k, r in rates.items()})
assert k_dev == F.K_FROZEN, (k_dev, F.K_FROZEN)
K = F.K_FROZEN
print("k =", K, "\n")

for name, value in F.run_checks(dev + test, K).items():
    print(f"{name}: {value}")

## 2. Exemplos few-shot (V5)

São **3 exemplos fixos do treino**, um por papel:

| Papel | Critério |
|---|---|
| `table_1step` | 1 passo, evidência só na tabela, resposta que **não** é porcentagem |
| `both_2step` | 2 passos, evidência na tabela **e** no texto, resposta em porcentagem |
| `text_only` | evidência só no texto (1 ou 2 passos) |

**Filtros dos candidatos** (função `fewshot_candidates`): resposta numérica; programa só com soma, subtração, multiplicação e divisão; o programa executado reproduz `exe_ans`; a resposta dos anotadores bate com `exe_ans` na métrica estrita; todos os operandos aparecem no contexto; contexto com até 3.000 caracteres (quartil mais curto do treino, para o prompt não crescer demais); tabela com mais de *k* linhas nos dois primeiros papéis; e a seleção de linhas preserva o gabarito do exemplo (para que a resposta seja dedutível em todas as condições).

O sorteio usa `FEWSHOT_SEED`, e os IDs ficaram fixos no módulo; nenhum exemplo foi escolhido à mão. Houve um sorteio anterior, sem o último filtro — a seção 3.1 da metodologia registra isso. O raciocínio de cada exemplo foi escrito à mão, é o mesmo nas seis condições e não cita o formato da tabela. **Só a tabela dos exemplos muda**: ela é serializada na representação da condição.

In [ ]:
print("Candidatos no treino por papel:",
      {kind: len(F.fewshot_candidates(train, kind, K)) for kind in F.FEWSHOT_KINDS})
print("Semente do sorteio:", F.FEWSHOT_SEED, "\n")

shots = F.get_fewshot(train)
for kind, shot in zip(F.FEWSHOT_KINDS, shots):
    qa = shot["qa"]
    print(f"[{kind}] {shot['id']}")
    print("  pergunta:", qa["question"])
    print("  programa:", qa["program"], "| exe_ans:", qa["exe_ans"],
          "| passos:", F.count_steps(qa["program"]), "| evidência:", F.evidence_source(qa["gold_inds"]))
    print("  resposta do exemplo:")
    for line in F.FEWSHOT_SOLUTIONS[shot["id"]].split("\n"):
        print("     ", line)
    print()

In [ ]:
# V5 — few-shot coerente
for name, value in F.check_fewshot(train, dev, test, K).items():
    print(f"{name}: {value}")

## 3. O prompt

Uma única mensagem de usuário, igual para os dois modelos, enviada no *chat template* oficial de cada um:

```
<instrução>

=== Example 1 ===
Report:
<contexto do exemplo, com a tabela na representação da condição>

Question: <pergunta do exemplo>

<raciocínio curto>
Answer: <resposta>

=== Example 2 ===  ...   === Example 3 ===  ...

=== Now answer this one ===
Report:
<contexto da pergunta>

Question: <pergunta>

<lembrete de formato>
```

O **lembrete de formato** é a última linha do prompt (`F.FORMAT_REMINDER`). Ele entrou depois da primeira rodada na GPU: sem ele, o Phi-4-mini terminava parte das respostas sem a linha `Answer:`. O texto foi escolhido entre três variantes no dev, fora das perguntas do piloto (seção 3 da metodologia).

Abaixo, o prompt completo de uma pergunta do dev em R1 e, em seguida, só o bloco da tabela dessa pergunta nas outras condições.

In [ ]:
probe = dev[0]
print(F.build_prompt(probe, "markdown", shots, K))

In [ ]:
for rep in F.REPRESENTATIONS:
    print(f"===== {F.REP_CODE[rep]} {rep} =====")
    print(F.serialize(probe, rep, K), "\n")

PROMPT_HASH = F.prompt_hash(K)
FINGERPRINT = F.prompts_fingerprint(test, shots, K)
print("Hash do template:", PROMPT_HASH)
print("Hash de todos os prompts do teste (1.147 × 6):", FINGERPRINT)

## 4. Tokens totais de entrada

O notebook 01 contou os tokens do **contexto** (relatório + tabela). Aqui contamos o **prompt inteiro**, já com instrução, exemplos e *chat template* — o custo real de cada chamada. Como as tabelas dos exemplos também estão na representação da condição, a diferença entre condições é maior no prompt do que no contexto.

In [ ]:
tokenizers = {}
for name in MODELS_TO_RUN:
    try:
        tokenizers[name] = F.load_tokenizer(name, HF_TOKEN)
        print(f"OK     {name}")
    except Exception as e:
        print(f"PULADO {name}: {type(e).__name__} — {str(e).splitlines()[0][:100]}")

rows = []
for ex in test:
    for rep in F.REPRESENTATIONS:
        prompt, context = F.build_prompt(ex, rep, shots, K), F.build_context(ex, rep, K)
        for name, tok in tokenizers.items():
            rows.append({"id": ex["id"], "model": name, "rep": rep, "code": F.REP_CODE[rep],
                         "prompt_tokens": F.count_tokens(tok, F.chat_text(tok, name, prompt)),
                         "context_tokens": F.count_tokens(tok, context)})
tokens_df = pd.DataFrame(rows)
tokens_df["prompt_hash"] = PROMPT_HASH
TOKENS_PATH = RESULTS_DIR / "tokens_prompt_test.csv"
if TOKENS_PATH.exists():                 # mantém a contagem de modelos que não puderam ser carregados nesta sessão,
    kept = pd.read_csv(TOKENS_PATH)      # desde que seja do MESMO prompt
    kept = kept[~kept["model"].isin(tokenizers) & (kept.get("prompt_hash") == PROMPT_HASH)]
    if len(kept):
        print("Contagem mantida do arquivo anterior para:", sorted(kept["model"].unique()))
        tokens_df = pd.concat([kept, tokens_df], ignore_index=True)
tokens_df.to_csv(TOKENS_PATH, index=False)

mean_tok = tokens_df.pivot_table(index="code", columns="model", values="prompt_tokens", aggfunc="mean")
print("\nMédia de tokens de entrada por pergunta (prompt inteiro, teste):\n")
print(mean_tok.round(0).astype(int).to_string())
print("\nRelativo a R1:\n")
print((mean_tok / mean_tok.loc["R1"]).round(3).to_string())
print("\nMaior prompt por modelo:", tokens_df.groupby("model")["prompt_tokens"].max().to_dict(),
      f"(+ {F.MAX_NEW_TOKENS} tokens de saída)")

## 5. Piloto no dev

Para cada modelo: carrega, gera as respostas e libera a memória. Os resultados vão para `pilot_dev/<modelo>__<representação>.jsonl`, gravados a cada lote, com retomada — se a sessão cair, basta rodar o notebook de novo: o que já está no Drive é pulado, e um modelo com o piloto completo nem é carregado.

Se o prompt for alterado depois de um piloto, os arquivos antigos são recusados (o hash do prompt muda): apague a pasta `pilot_dev/` e rode de novo.

In [ ]:
pilot = random.Random(PILOT_SEED).sample(sorted(dev, key=lambda e: e["id"]), PILOT_N)
pilot_small = pilot[:PILOT_OTHER_REPS_N]
REVISIONS_PATH = RESULTS_DIR / "pilot_dev" / "revisions.json"
revisions = F.load_json(REVISIONS_PATH) if REVISIONS_PATH.exists() else {}
PILOT_SIZE = {rep: len(pilot if rep == "markdown" else pilot_small) for rep in F.REPRESENTATIONS}

# Nada é gerado se a pasta tem piloto de outro prompt ou de outra precisão: continuar misturaria os dois
other = F.pilot_state(RESULTS_DIR) - {(PROMPT_HASH, PRECISION)}
assert not other, (
    f"A pasta {RESULTS_DIR / 'pilot_dev'} tem piloto gerado com outro prompt ou outra precisão: {sorted(other)}. "
    f"O atual é {(PROMPT_HASH, PRECISION)}. MOVA essa pasta para outro nome (não apague) e, se já existe o piloto do "
    f"prompt atual em outra pasta, renomeie-a para 'pilot_dev'. Nenhuma geração foi feita.")

print("Andamento do piloto antes desta sessão (perguntas já gravadas):")
print(F.progress(RESULTS_DIR, "pilot_dev", MODELS_TO_RUN, PILOT_SIZE).to_string(), "\n")

for name in MODELS_TO_RUN:
    pending = [rep for rep in F.REPRESENTATIONS
               if len(F.read_jsonl(F.result_path(RESULTS_DIR, "pilot_dev", name, rep))) < PILOT_SIZE[rep]]
    if not pending and name in revisions:
        print(f"{name}: piloto completo, nada a fazer.")
        continue
    assert ALLOW_NEW_PILOT, (
        f"O piloto de {name} não está completo em {RESULTS_DIR / 'pilot_dev'} (faltam: {pending or 'revisão do modelo'}). "
        f"O piloto do estudo JÁ FOI FEITO: o mais provável é que a pasta do Drive montada seja outra com o mesmo nome, ou que a "
        f"pasta do piloto ainda esteja com outro nome (por exemplo, 'pilot_dev_old_bd940bbf'). Nenhuma geração foi feita. "
        f"Para gerar um piloto novo de propósito, defina ALLOW_NEW_PILOT = True na célula de configuração.")
    assert HAS_GPU, f"Falta gerar o piloto de {name}: troque o runtime para um com GPU e rode o notebook de novo."
    assert PRECISION != "bf16" or GPU_GB >= 22, (
        f"Falta gerar o piloto de {name} em bf16, que não cabe nesta GPU ({GPU_GB:.0f} GB). Use L4, A100 ou G4.")
    try:
        tok, model, revision = F.load_model(name, PRECISION, HF_TOKEN, revisions.get(name))
    except Exception as e:
        print(f"PULADO {name}: {type(e).__name__} — {str(e).splitlines()[0][:150]}")
        continue
    revisions[name] = revision
    F.save_json(REVISIONS_PATH, revisions)
    for rep in pending:
        subset = pilot if rep == "markdown" else pilot_small
        F.run_condition(tok, model, name, subset, rep, shots,
                        F.result_path(RESULTS_DIR, "pilot_dev", name, rep),
                        split="pilot_dev", precision=PRECISION, k=K, batch_size=BATCH_SIZE,
                        save_every=SAVE_EVERY)
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

print("\nAndamento do piloto depois desta sessão:")
print(F.progress(RESULTS_DIR, "pilot_dev", MODELS_TO_RUN, PILOT_SIZE).to_string())

## 6. Relatório do piloto (V7)

Por modelo × representação:

- **desfecho da saída** (seção 6.3 da metodologia) — cada saída cai em exatamente um:
  - `resposta`: resposta legível na linha do `Answer:` (só estas podem ser acertos);
  - `recusa`: o modelo diz que não dá para responder (`Answer: no` em pergunta numérica, `Answer: cannot be determined`);
  - `outra`: há `Answer:`, seguido de algo que não é resposta nem recusa;
  - `cortada`: bateu no limite de tokens antes de escrever `Answer:` (em geral, um laço de repetição);
  - `sem_answer`: terminou sozinha sem a linha `Answer:`. É a única **falha de formato do prompt**;
- **V7**, em R1 e para cada modelo: (a) `sem_answer` abaixo de 5% e (b) `resposta` em pelo menos 90%. Recusas e cortes são comportamento do modelo: contam como erro na acurácia e são relatados, mas não reprovam o prompt enquanto (b) valer;
- **estrita / tolerante** — acurácia no piloto (só para ver se o prompt funciona; não é resultado do estudo);
- **cortadas** — saídas que bateram no limite de tokens sem terminar;
- **mais de um `Answer:`** e **nova pergunta depois da resposta** — o parser lê o *último* `Answer:`. Se o modelo continua escrevendo e inventa outra pergunta, a saída tem formato válido, mas o número avaliado é o errado. Se isso for frequente, o prompt precisa de ajuste **aqui, no dev**.

In [ ]:
pil = F.load_results(RESULTS_DIR, "pilot_dev")
assert len(pil), "Nenhum resultado de piloto encontrado: rode a seção 5."
outside = sorted(set(pil["model"]) - set(F.MODELS))
if outside:
    print(f"Arquivos de modelos fora do estudo, ignorados no relatório e no congelamento: {outside}\n")
    pil = pil[pil["model"].isin(F.MODELS)].copy()
assert len(pil), "Não há piloto dos modelos do estudo: rode a seção 5."

# Só vale piloto gerado com o prompt e a precisão ATUAIS (inclusive de modelos que não rodaram nesta sessão)
if PRECISION is None and pil["precision"].nunique() == 1:
    PRECISION = pil["precision"].iloc[0]          # sessão sem GPU, só para reler um piloto já feito
stale = pil[(pil["prompt_hash"] != PROMPT_HASH) | (pil["precision"] != PRECISION)]
assert stale.empty, (f"Há piloto antigo de {sorted(stale['model'].unique())}, gerado com outro prompt ou outra "
                     f"precisão. Mova esses arquivos de {RESULTS_DIR / 'pilot_dev'} para outra pasta e rode de novo.")

# O relatório avalia a saída bruta com o parser ATUAL (o piloto é onde o parser é validado; os campos
# gravados no JSONL são os do momento da geração)
dev_by_id = {ex["id"]: ex for ex in dev}
now = pd.DataFrame([F.evaluate(raw, dev_by_id[i]["qa"]["exe_ans"]) for raw, i in zip(pil["raw_output"], pil["id"])],
                   index=pil.index)
changed = int(((now["format_ok"] != pil["format_ok"]) | (now["strict"] != pil["correct_strict"])
               | (now["lenient"] != pil["correct_lenient"])).sum())
if changed:
    print(f"Aviso: {changed} registros foram gravados com outra versão do parser; o relatório usa a atual.\n")
pil["format_ok"], pil["correct_strict"], pil["correct_lenient"], pil["parsed_answer"] = \
    now["format_ok"], now["strict"], now["lenient"], now["pred"]

pil = pd.concat([pil, pd.DataFrame(list(pil["raw_output"].map(F.output_flags)), index=pil.index)], axis=1)

outcomes = F.outcome_rates(pil, dev)
print("Desfecho das saídas (fração por modelo × condição):\n")
print(outcomes.round(3).to_string(), "\n")

pilot_summary = pil.groupby(["model", "code"]).agg(
    n=("id", "size"),
    estrita=("correct_strict", "mean"),
    tolerante=("correct_lenient", "mean"),
    tokens_saida=("output_tokens", "mean"),
    cortadas=("hit_max_new_tokens", "mean"),
    mais_de_um_answer=("n_answer_marks", lambda s: (s > 1).mean()),
    nova_pergunta=("new_item_after_answer", "mean"),
    s_por_pergunta=("latency_s", "mean"),
)
print(pilot_summary.round(3).to_string())

print(f"\nV7 em R1 — (a) sem linha 'Answer:' < {V7_MAX_NO_ANSWER:.0%}  e  (b) com resposta legível >= {V7_MIN_ANSWERED:.0%}")
v7 = {}
for name in sorted(pil["model"].unique()):
    r1 = pil[(pil["model"] == name) & (pil["rep"] == "markdown")]
    complete = len(r1) == PILOT_N
    row = outcomes.loc[(name, "R1")] if (name, "R1") in outcomes.index else None
    no_answer = float(row["sem_answer"]) if row is not None else float("nan")
    answered = float(row["resposta"]) if row is not None else float("nan")
    v7[name] = bool(complete and no_answer < V7_MAX_NO_ANSWER and answered >= V7_MIN_ANSWERED)
    status = "PASSA" if v7[name] else ("INCOMPLETO" if not complete else "FALHA")
    print(f"  {name:12s} sem Answer: {no_answer:5.1%} | com resposta: {answered:5.1%} | recusa {float(row['recusa']) if row is not None else 0:5.1%} | "
          f"cortada {float(row['cortada']) if row is not None else 0:5.1%}  ({len(r1)}/{PILOT_N} perguntas)  -> {status}")

runaway = pil.groupby("model")["new_item_after_answer"].mean()
for name, rate in runaway.items():
    if rate > 0.02:
        print(f"\nATENÇÃO: {name} continuou com uma nova pergunta depois da resposta em {rate:.1%} das saídas.")

In [ ]:
# Amostra de saídas: três de R1 por modelo e até cinco falhas de formato por modelo
for name in sorted(pil["model"].unique()):
    sub = pil[(pil["model"] == name) & (pil["rep"] == "markdown")]
    print("=" * 100, f"\n{name} — exemplos em R1")
    for rec in sub.head(3).itertuples():
        print(f"\n[{rec.id}] estrita={rec.correct_strict} | extraído={rec.parsed_answer}")
        print(rec.raw_output)
    bad = pil[(pil["model"] == name) & (~pil["format_ok"])]
    print(f"\n{name} — falhas de formato ({len(bad)} no piloto)")
    for rec in bad.head(5).itertuples():
        print(f"\n[{rec.id} | {rec.code}] {rec.raw_output[-300:]!r}")

## 7. Tempo estimado para o teste

Projeção a partir da latência média do piloto, por modelo: 1.147 perguntas × 6 condições. Para as representações com poucas perguntas no piloto a estimativa é mais grosseira.

In [ ]:
lat = pil.groupby(["model", "rep"])["latency_s"].mean().unstack("rep")
hours = (lat * len(test)).sum(axis=1) / 3600
print("Horas estimadas por modelo (teste completo, 6 condições):")
print(hours.round(1).to_string())
print(f"\nTotal: {hours.sum():.1f} h de GPU. O notebook 03 retoma de onde parou, então pode ser dividido em várias sessões.")

## 8. Congelamento

Grava `frozen_config.json` com tudo o que o notebook 03 precisa manter fixo: hash do prompt, hash de todos os prompts do teste, exemplos, *k*, precisão, tamanho do lote e a revisão de cada modelo. Só entram os modelos que **passaram na V7**.

- Se algum modelo falhou na V7: ajuste o prompt em `src/finrag.py`, **faça commit** da mudança, registre-a em `docs/METODOLOGIA.md`, mova `pilot_dev/` para outra pasta e rode este notebook de novo. Um piloto gerado com código que não está no repositório não é reproduzível e não serve para congelar.
- Para incluir depois um modelo que ficou de fora, acrescente-o a `MODELS` em `src/finrag.py`, faça commit e rode o piloto dele: se o prompt e a precisão forem os mesmos, ele é **acrescentado** ao congelamento.
- Depois que existir qualquer resultado do teste, o congelamento **não pode mais mudar**.

In [ ]:
passed = [name for name, ok in v7.items() if ok]
left_out = [name for name in F.MODELS if name not in passed]
assert passed, "Nenhum modelo passou na V7: ajuste o prompt no dev antes de congelar."
print("Entram no congelamento:", passed)
if left_out:
    print("FICAM DE FORA (sem piloto completo ou reprovados na V7):", left_out)

core = {
    "prompt_hash": PROMPT_HASH,
    "prompts_fingerprint_test": FINGERPRINT,
    "fewshot_ids": list(F.FEWSHOT_IDS),
    "fewshot_seed": F.FEWSHOT_SEED,
    "k": K,
    "max_new_tokens": F.MAX_NEW_TOKENS,
    "precision": PRECISION,
    "batch_size": BATCH_SIZE,
}
models = {}
if FROZEN_PATH.exists():
    old = F.load_json(FROZEN_PATH)
    same = all(old.get(key) == value for key, value in core.items())
    has_test = any((RESULTS_DIR / "test").glob("*__*.jsonl"))
    assert same or not has_test, ("Já existem resultados do teste gerados com outro congelamento. "
                                  "O prompt e a configuração não podem mais mudar.")
    if same:
        models = old["models"]
    else:
        print("Congelamento anterior substituído (ainda não havia resultados do teste).")

for name in passed:
    r1 = pil[(pil["model"] == name) & (pil["rep"] == "markdown")]
    models.setdefault(name, {
        "repo": F.MODELS[name],
        "revision": revisions.get(name),
        "pilot_outcomes_r1": {kind: round(float(outcomes.loc[(name, "R1"), kind]), 4) for kind in F.OUTCOMES},
        "pilot_strict_r1": round(float(r1["correct_strict"].mean()), 4),
        "pilot_n": int(len(r1)),
    })

frozen = dict(core, models=models,
              versions={"python": platform.python_version(), "torch": torch.__version__,
                        "transformers": transformers.__version__},
              gpu=GPU_NAME, git_sha=F.git_sha(REPO), git_dirty=F.git_dirty(REPO),
              frozen_at=datetime.datetime.now().isoformat(timespec="seconds"))
F.save_json(FROZEN_PATH, frozen)
print("Congelado em", FROZEN_PATH, "\n")
print(json.dumps(frozen, ensure_ascii=False, indent=2))

## Próximo passo

Com o `frozen_config.json` gravado, o **notebook 03** roda o teste (1.147 perguntas × 6 condições × modelos congelados) e o **notebook 04** faz a análise.